In [ ]:
# Intiating Code Setting up physical constants
import numpy as np
import math
import random
import matplotlib.pyplot as plt


## Important constants
particle_names = ['proton', 'electron', 'pion', 'muon', 'neutrino']

## Unit conversions
km_per_ls = 2.997e5

In [ ]:
def run_cosmic_MC(particles, dt, Nsteps, attenuation=True):
    """
    Main loop for cosmic ray Monte Carlo project.

    Arguments:
    =====
    * particles: a list of any length, containing "particle" tuples.
        A particle tuple has the form:
            (name, p_x, p_y, traj_x, traj_y)
        where p_x and p_y are momentum vector components, and
        traj_x and traj_y are NumPy arrays of distance.

    * dt: time between Monte Carlo steps, in seconds.
    * Nsteps: Number of steps to run the Monte Carlo before returning.
    * attenuation: if True, protons and muons with too little energy are pruned
      from the simulation.  (This is a crude simulation of the real-world effect
      that charged particles moving through the air will lose energy.)

    Returns:
    =====
    A list of particle tuples.

    Example usage:
    =====

    >> init_p_x, init_p_y = velocity_to_momentum(particle_mass('muon'), 0.85, -0.24) # beta ~ 0.8, relativistic
    >> init_particles = [ ('muon', init_p_x, init_p_y, np.array([0]), np.array([1e-4]) ) ]  # ~ 30 km height in light-sec
    >> particles = run_cosmic_MC(init_particles, 1e-5, 100)

    The 'particles' variable after running should contain three particle tuples: the
    initial muon, an electron, and a neutrino.  (Even though the muon decays,
    we keep it in the final particle list for its trajectory.)

    """

    E_threshold = 2000 # MeV - change this constant if your unit system is different!!

    stopped_particles = []
    for step_i in range(Nsteps):
        updated_particles = []

        for particle in particles:
            # Unpack particle tuple
            (name, p_x, p_y, traj_x, traj_y) = particle
            if traj_y[-1] <= 0:
                stopped_particles.append(particle)
                continue
            (beta_x, beta_y) = momentum_to_velocity(particle_mass(name), p_x, p_y)

            # Check for interaction
            does_interact = check_for_interaction(particle, dt, traj_y[-1])

            if does_interact:
                if name == 'proton':
                    decay_products = proton_downscatter(beta_x, beta_y)
                else:
                    stopped_particles.append(particle)
                    decay_products = particle_decay(name)

                # Transform products back to lab frame
                for product in decay_products:
                    (product_name, product_p_x, product_p_y) = boost_product_to_lab(product, beta_x, beta_y)

                    ## Clean up - attentuation of particles with too little energy
                    if (attenuation):
                        E_product = np.sqrt(product_p_x**2 + product_p_y**2 + particle_mass(product_name)**2)
                        if E_product < E_threshold:
                            if product_name == 'proton':
                                # Avoid losing proton tracks, since downscatter doesn't add stopped particles
                                stopped_particles.append(particle)
                            if product_name != 'neutrino':
                                # Skip decay product, don't append to updated_particles
                                continue

                    # If this was a proton scatter, then the "new" proton is
                    # the same as the original, so keep track of its trajectory!
                    if name == 'proton' and product_name == 'proton':
                        product_traj_x = traj_x
                        product_traj_y = traj_y
                    else:
                        product_traj_x = np.array([traj_x[-1]])
                        product_traj_y = np.array([traj_y[-1]])

                    # Make new particle tuple and append
                    product_particle = (product_name, product_p_x, product_p_y,
                                        product_traj_x, product_traj_y)
                    updated_particles.append( product_particle )
            else:
                # Doesn't interact, so compute motion
                traj_x = np.append(traj_x, traj_x[-1] + beta_x * dt)
                traj_y = np.append(traj_y, traj_y[-1] + beta_y * dt)

                updated_particles.append( (name, p_x, p_y, traj_x, traj_y) )


        # Run next timestep
        particles = updated_particles

    # Add stopped particles back to list and return
    particles.extend(stopped_particles)
    return particles

In [ ]:
def lorentz_beta_gamma(beta_x, beta_y):
    '''
    Given the measured x and y components of beta, returns the Lorentz factors beta and gamma

    Args:
      - beta_x: The x component of the velocity beta
      - beta_y: The y component of the velocity beta

    Return:
      - beta: The lorentz factor beta
      - gamma: The lorentz factor gamma
    '''

    # Compute beta
    beta = np.sqrt(beta_x**2 + beta_y**2)

    # Verify beta is no greater than one as that is not possible
    if beta > 1:
        raise ValueError("Beta cannot be > 1!")

    # Compute gamma
    gamma = 1/np.sqrt(1-beta**2)


    return (beta, gamma)

# Compute new momentum components, following a boost to a new frame given by (beta_x, beta_y)
# Note: you might need to treat the massless case separately!
def boost_momentum(m, px, py, beta_x, beta_y):
    '''
    Given the measured x and y components of the momentum and velocity, returns the x and y components of the momentum adjusted to the a new fram of referance

    Args:
      - beta_x: The x component of the velocity of the new frame
      - beta_y: The y component of the velocity of the new frame
      - px: The x component of the momentum of the particle
      - py: The y component of the momentum of the particle
      - m: the mass of the particle

    Return:
      - px_boost: The boosted x component of the momentum of the particle
      - py_boost: The boosted y component of the momentum of the particle
    '''
    # Compute beta and gamma using lorentz_beta_gamma
    beta, gamma = lorentz_beta_gamma(beta_x, beta_y)

    # Check edge case of no motion
    if beta == 0:
      return(px, py)

    #Compute the energy for the mass and massless case
    p = np.sqrt(px**2 + py**2)
    if m == 0:
      E = p
    else:
      E = np.sqrt(p**2 + m**2)

    #Compute the boost
    px_boost = px - (gamma * E * beta_x) + ((gamma - 1) * beta_x * ((beta_x*px + beta_y*py)/beta**2))

    py_boost = py - (gamma * E * beta_y) + ((gamma - 1) * beta_y * ((beta_x*px + beta_y*py)/beta**2))

    return (px_boost, py_boost)

# Given a particle with mass m, convert momentum vector to velocity vector
def momentum_to_velocity(m, px, py):
  """
    Given a particle with mass m, convert momentum vector
    to velocity vector.

    Args:
      - m: mass of particle, in MeV/c^2
      - (px, py): momentum of particle, in MeV/c

    Returns:
      - (beta_x, beta_y): velocity vector of particle
    """
  #Calculate the total momentum and handle the edge case
  p = np.sqrt(px**2 + py**2)
  if p == 0:
    return (0,0)

  # Massless case
  if m == 0:
    beta_x = px/p
    beta_y = py/p

  # Mass case
  else:
    beta_x = (px/m)/np.sqrt(1+(p/m)**2)
    beta_y = (py/m)/np.sqrt(1+(p/m)**2)

  return (beta_x, beta_y)

# Opposite of momentum_to_velocity: given mass and velocity, compute the momentum vector
def velocity_to_momentum(m, beta_x, beta_y):
  """
    Given a particle with mass m, convert velocity vector
    to momentum vector.

    Args:
      - m: mass of particle, in MeV/c^2
      - (beta_x, beta_y): velocity of particle

    Returns:
      - (px, py): momentum vector of particle
  """

  beta, gamma = lorentz_beta_gamma(beta_x, beta_y)

  # Massless case
  if m == 0:
    raise ValueError("No Mass")

  # Mass case
  else:
    px = gamma * beta_x * m
    py = gamma * beta_y * m

  return (px, py)


def check_for_interaction(particle, dt, h):
    """
    Given a particle tuple a timestep and the height of a particle, returns a boolean value for whether the particle will decay and scatter.

    Args:
      - particle: a tuple containing the name and x and y components of a particle
      - dt: the timestep over which a particle could interact
      - h: the height that the particle is at

    Returns:
      - np.random.rand() < p: A boolean (True/False) value of a random number being within the probability p of the particle interacting
    """

    # Initial variables pulled from the particle tuple
    name = particle[0]
    px = particle[1]
    py = particle[2]
    p = np.sqrt(px**2 + py**2)
    mass = particle_mass(name)

    #Automatically returns false if the paricle has no chance of interaction
    if name not in ['proton', 'pion', 'muon']:
      return False

    #Calculate the energy of the particle, and if its a proton check that it is above the threshold energy
    E = np.sqrt(p**2 + mass**2)
    if name == 'proton' and E < 1218:
      return False

    #Calculate the velocity and use it to find the lorentz factors
    betax, betay = momentum_to_velocity(mass, px, py)
    beta, gamma = lorentz_beta_gamma(betax, betay)


    # Time constants from cosmic_assign
    tau_p = np.exp((h * km_per_ls)/7) * (1/beta) * 1.14e-5 # s
    tau_pi = gamma * 2.603e-8 # s
    tau_mu = gamma * 2.197e-6 # s


    # Consider cases for different particles
    if name == 'proton':
      tau = tau_p
    elif name == 'pion':
      tau = tau_pi
    elif name == 'muon':
      tau = tau_mu

    # Compute probability of interaction
    p = 1 - np.exp(-dt/tau)

    # Interaction happens with probability p
    return np.random.rand() < p

# Simulate the decay X --> YZ of a pion or muon. Here, Y is a muon (electron), and Z is a neutrino
def particle_decay(name):
  """
    Given the name of a particle return two tuples that are the products of that particles decay.

    Args:
      - name: the name of the particle

    Returns:
      - (particle_Y, particle_Z): two tuples containg the name and momentum components of the decay products
  """
  # consider pion and muon case and set product names
  if name == 'pion':
      Yname = 'muon'

  elif name == 'muon':
      Yname = 'electron'

  else:
    return()
  Zname = 'neutrino'

  #Select a random angle for the scatter
  theta = random.uniform(0, 2*np.pi)

  #Calculate the mass of the new particles
  Mx = particle_mass(name)
  My = particle_mass(Yname)
  M = Mx**2 - My**2

  #Calculate the new momentum components of the product particles
  p = M/(2*Mx)
  px = p * np.cos(theta)
  py = p * np.sin(theta)

  #Compile results and return
  particle_Y = (Yname, px, py)
  particle_Z = (Zname, -px, -py)
  return (particle_Y, particle_Z)

# Simulate the downscattering of a proton off of an air molecule, p --> p + pi + pi
def proton_downscatter(beta_x, beta_y):
  '''
    Simulate the full downscatter of a proton given the the components of the velocity of the proton and returns the products after the collision of the proton and Nitrogen-14 atom.

    Args:
      - beta_x: x component of the protons velocity
      - beta_y: y component of the protons velocity

    Returns:
      - particle_pi_1: the tuple containg the name and momentum of the first pion product
      - particle_pi_2: the tuple containg the name and momentum of the second pion product
      - particle_p: the tuple containg the name and momentum of the proton product
  '''

  # initial variables
  Mpi = particle_mass('pion')
  Mn = 13040
  b = 1/500
  c = 1
  beta, gamma = lorentz_beta_gamma(beta_x, beta_y)

  # Initial Nitrogen-14 state
  Pn_x = -gamma * Mn * beta_x
  Pn_y = -gamma * Mn * beta_y
  Pn = np.sqrt(Pn_x**2 + Pn_y**2)
  En = np.sqrt((Pn**2 * c**2) + (Mn**2 * c**4))

  # Calculate the energy and momentum of the first pion
  y1 = random.uniform(0,1)
  Epi_1 = (Mpi * c**2) - (1/b)*np.log(1 - y1)
  Ppi_1 = np.sqrt(Epi_1**2 - (Mpi**2 * c**4))/c

  theta1 = random.uniform(0, 2 * np.pi)
  Ppi1_x = Ppi_1 * np.cos(theta1)
  Ppi1_y = Ppi_1 * np.sin(theta1)

  # Calculate the energy and momentum of the second pion
  y2 = random.uniform(0,1)
  Epi_2 = (Mpi * c**2) - (1/b)*np.log(1 - y2)
  Ppi_2 = np.sqrt(Epi_2**2 - (Mpi**2 * c**4))/c

  theta2 = random.uniform(0, 2 * np.pi)
  Ppi2_x = Ppi_2 * np.cos(theta2)
  Ppi2_y = Ppi_2 * np.sin(theta2)

  # Energy after colision
  En_prime = En - Epi_1 - Epi_2

  # Momentum after colision
  Pn_prime = (1/c) * np.sqrt(En_prime**2 - (Mn**2 * c**4))

  # Change in momentum
  delta_Pnx = Pn_x * (1 - (Pn_prime/Pn))
  delta_Pny = Pn_y * (1 - (Pn_prime/Pn))

  # Proton Momentum
  Pp_x = delta_Pnx - Ppi1_x - Ppi2_x
  Pp_y = delta_Pny - Ppi1_y - Ppi2_y

  particle_pi_1 = ('pion', Ppi1_x, Ppi1_y)
  particle_pi_2 = ('pion', Ppi2_x, Ppi2_y)
  particle_p = ('proton', Pp_x, Pp_y)


  return (particle_pi_1, particle_pi_2, particle_p)

# Given a decay product tuple (name, px, py)
# and the two components of velocity (beta_x, beta_y) of the parent particle,
# returns the tuple (name, px_lab, py_lab) giving the product's momentum
# in the lab frame, i.e. after boosting the momentum by (-beta_x, -beta_y)
def boost_product_to_lab(product, beta_x, beta_y):
  '''
    Given a particle tuple and the components of the velocity, return the tuple with the product name and momentum components of the particle after being boosted into the lab frame of reference.

    Args:
      - product: The product tuple containing the name and momentum of the product particle
      - (beta_x, beta_y): velocity of particle

    Returns:
      - (product_name, px_lab, py_lab): The product tuple containing the name and boosted momentum of the particle
  '''

  #Collect the intial values from the product tuple
  product_name = product[0]
  px = product[1]
  py = product[2]
  mass = particle_mass(product_name)

  #Calculate the boosted momentums using boost_momentum
  px_lab, py_lab = boost_momentum(mass, px, py, -beta_x, -beta_y)

  return (product_name, px_lab, py_lab)

# Given a particle name, return its mass (in whatever units you are using.)
def particle_mass(name):
  '''
    Given the name of a particle return the mass of that particle in MeV/c^2.

    Args:
      - name: name of the particle

    Return:
      - mass: the mass of the particle in MeV/c^2
  '''
  if name == 'proton':
    mass = 938.3 #MeV/c^2
  elif name == 'electron':
    mass = 0.511 #MeV/c^2
  elif name == 'pion':
    mass = 139.6 #MeV/c^2
  elif name == 'muon':
    mass = 105.7 #MeV/c^2
  elif name == 'neutrino':
    mass = 0.0 #MeV/c^2
  return mass

In [ ]:
# --- Test Parameters ---
beta_x = 0.0
beta_y = -0.995
starting_height = 1e-4 # ~30 km up, in light-seconds
dt = 1e-6              # Small timestep
Nsteps = 500           # Enough steps to reach the ground

# --- Proton Test Parameters ---
beta_x_p = 0.95999
beta_y_p = -0.28
starting_height_p = 20.0/km_per_ls

# Calculate initial momenta
pe_x, pe_y = velocity_to_momentum(particle_mass('electron'), beta_x, beta_y)
pmu_x, pmu_y = velocity_to_momentum(particle_mass('muon'), beta_x, beta_y)
ppi_x, ppi_y = velocity_to_momentum(particle_mass('pion'), beta_x, beta_y)
pp_x, pp_y = velocity_to_momentum(particle_mass('proton'), beta_x_p, beta_y_p)

# Initialize the separate single-particle showers
shower_e = [('electron', pe_x, pe_y, np.array([0.0]), np.array([starting_height]))]
shower_mu = [('muon', pmu_x, pmu_y, np.array([0.0]), np.array([starting_height]))]
shower_pi = [('pion', ppi_x, ppi_y, np.array([0.0]), np.array([starting_height]))]
shower_p = [('proton', pp_x, pp_y, np.array([0.0]), np.array([starting_height_p]))]

# Run the MC (attenuation = False as requested)
results_e = run_cosmic_MC(shower_e, dt, Nsteps, attenuation=False)
results_mu = run_cosmic_MC(shower_mu, dt, Nsteps, attenuation=False)
results_pi = run_cosmic_MC(shower_pi, dt, Nsteps, attenuation=False)
results_p = run_cosmic_MC(shower_p, dt, Nsteps, attenuation=False)

# --- Plotting ---
# (sharey=True is removed because the proton starts at 20, which would squish the other plots)
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
titles = ['Electron Initial', 'Muon Initial', 'Pion Initial', 'Proton Initial']
results = [results_e, results_mu, results_pi, results_p]

color_map = {'proton': 'red', 'pion': 'blue', 'muon': 'green', 'electron': 'orange', 'neutrino': 'gray'}

for ax, res, title in zip(axes, results, titles):
    plotted_labels = set()
    for particle in res:
        name, _, _, traj_x, traj_y = particle
        label = name if name not in plotted_labels else None
        plotted_labels.add(name)

        line_style = '--' if name == 'neutrino' else '-'
        line_width = 1.0 if name == 'neutrino' else 1.5

        ax.plot(traj_x, traj_y, label=label, color=color_map.get(name, 'black'),
                linestyle=line_style, linewidth=line_width, alpha=0.7)

    ax.set_title(title)
    ax.set_xlabel("x (light-seconds)")
    ax.set_ylabel("y (Altitude in light-seconds)")
    ax.grid(True, linestyle=':', alpha=0.7)
    ax.legend(loc='upper right')

    # Only enforce the tight x-axis bounds on the vertically-falling particles
    if title != 'Proton Initial':
        ax.set_xlim(-2e-5, 2e-5)

plt.tight_layout()
plt.show()